### Codetable代码表
在每个HDB数据文件中，对每个标的的静态基本信息，用hdb.Codetable对象来描述，该对象包括以下四个属性：

    symbols 				# 标的代码列表
    total_items_num       # 总数据条目数
    type_items_nums  		# 各种类型的数据条目数
    data 				   # 自定义数据，数据类型通过hdb.DataType对象描述，用于存放代码对应的静态基础信息。可以通过hdb.File.ci_type.items_data(data)解析


### DataItem数据记录
在HDB文件中，除了记录代码基础信息的代码表数据之外，其他数据都是异构的时序数据DataItem。DataItem中前五个字段是所有数据共有的，第六个字段因数据类型而异，其结构如下：

    symbol					# 标的代码
    trading_day             # 交易日
    local_time              # 本地时间戳，Unix时间,1970-1-1到当前时间经过的毫秒数
    index                 # 该代码在该数据文件全量代码表中的索引
    time_point_seq_no		# 该数据条目在该时间点的所有数据条目中的序号。在一个数据文件中，同一个时间点可能会有多条数据记录，我们通过该字段来标志这些记录的先后次序
    type_id       			# 数据类型id。即该数据类型在当前数据文件的数据 类型数组中的位置，系统生成参数
    data 					  # 自定义数据，数据类型通过hdb.DataType对象描述，可以通过hdb.File.data_types[type_id].items_data(data)解析 

### data自定义数据类型
data的数据类型由hdb.DataType来描述。该对象主要有两个属性：

    DataType.name       #数据类型名称。
    DataType.fields      #数据类型的所有字段组成的列表，列表元素为hdb.DataField对象，用于描述字段的详细属性。
    
#### 自定义数据类型（hdb.DataType）的构造方法
自定义数据类型可以通过  
```hdb.DataType(name, fields)
```
来创建一个自定义数据类型。  
参数：

    name     #数据类型名  
    fields    #字段列表，列表元素为hdb.DataField对象，用于描述该数据类型的各个字段的属性。

返回：hdb.DataType对象。  

例如：定义一个k线数据类型，类型取名为kline, 该数据类型包括“开盘价”、“收盘价”、“最高价”、“最低价”四个字段。利用hdb.DataField方法为每个字段生成一个hdb.DataField对象，将其添加进列表fields中，然后调用：
```hdb.DataType("kline",fields)```  
即可生成一个名为kline的自定义数据类型。  

#### 自定义数据类型（hdb.DataType）的解析方法
通常我们通过ReadTask.read()读取到的数据中，其data部分是一个地址，需要通过hdb.DataType.items_data()方法来解析。例如：
```
import hdb
db = hdb.DB(r"X:\hdb_data")
file = db.open_file(r"marketdata\tick_20220506", mode="r")
task = file.open_read_task(symbols=['SH.600000'], types=['SecurityTick'])
items = task.read()
data_type = file.data_types[items['type_id'][0]]
data = data_type.items_data(items['data'])
```
    

#### 自定义数据类型数组单个字段信息（hdb.DataField） 
自定义数据类型数组单个字段主要由hdb.DataField描述。其主要属性如下：


    DataField.name 				# 字段名
    DataField.type 				# 字段类型（支持的选项见hdb.FieldType）
    DataField.encode_op 		# 字段压缩操作（支持的选项见附录hdb.FieldEncodeOp）
    DataField.size 				# 数组长度
    DataField.flags 			# 额外压缩字段，1表示启用，0表示不启用。该字段为可选字段，即当新的一条记录中该字段的值相对于当前记录中该字段值保持不变时，新的记录中可在压缩编码时跳过该字段。一般除了我们确定基本上每条记录中都会更新的字段，比如时间字段，其它字段我们都可以设置该选项。

#### hdb.FieldType
   
|字段类型|说明|
|:-|:-|
|FieldType.Char|字符类型，长度1字节|
|FieldType.Short|短整型，长度2字节|
|FieldType.UShort|无符号短整型，长度2字节|
|FieldType.Int|整型，长度4字节|
|FieldType.UInt|无符号整型，长度4字节|
|FieldType.Long|长整型，长度8字节|
|FieldType.ULong|无符号长整型，长度8字节|
|FieldType.Float|单精度浮点数，长度4字节|
|FieldType.Double|双精度浮点数，长度8字节|
|FieldType.CharArray|字符数组|
|FieldType.ZeroTermCharArray|以0结尾的字符数组|
|FieldType.SpaceTermCharArray|以空格结尾的字符数组|
|FieldType.IntArray|整数数组|
|FieldType.ZeroTermIntArray|以0结尾的整数数组|
|FieldType.UIntArray|无符号整数数组|
|FieldType.ZeroTermUIntArray|以0结尾的无符号整数数组|
|FieldType.LongArray|长整数数组|
|FieldType.ZeroTermLongArray|以0结尾的长整数数组|
|FieldType.ULongArray|无符号长整数数组|
|FieldType.ZeroTermULongArray|以0结尾的无符号长整数数组|

####  hdb.FieldEncodeOp
  
    FieldEncodeOp.Raw    #不压缩，即直接拷贝该字段原始数据。
    FieldEncodeOp.ValueCompress  #值压缩，即对数值进行压缩(Varint编码)。注意，FieldType的Char与CharArray不支持该压缩编码方式。  
    FieldEncodeOp.ValueIncCompress #增量值压缩，即对该字段相对于上一条记录的增量数值进行压缩(Varint编码)。注意，FieldType的Char、CharArray、ZeroTermCharArray、SpaceTermCharArray不支持该压缩编码方式。


#### 字段信息（hdb.DataField）的构造方法
HDB作为一个自描述文件，任意数据类型的任意字段都由hdb.DataField对象来描述，该对象可以通过    
``` hdb.DataField(name, type, encode_op, size=0, flags=0)```
构造。  
 
 
例如：股票的k线数据至少包括“开盘价”、“收盘价”、“最高价”、“最低价”四个字段。以开盘价为例，字段名取为open，字段类型用无符号长整型，数组长度为一，则可以调用:  
```hdb.DataFiled('open',hdb.FieldType.ULong,hdb.FieldEncodeOp.ValueIncCompress,1,0)```   
来创建开盘价的描述字段。

 
## HDB读写(本地/远程连接服务器)功能相关API介绍
HDB读写本地和远程两种模式：本地模式读取本地HDB文件来获取行情；远程模式是指连接远端的服务器来获取数据。hdb模块有以下10个类
 
 + hdb.Client
 + hdb.Codetable
 + hdb.DB
 + hdb.DataField
 + hdb.DataFrame
 + hdb.File
 + hdb.ReadTask
 
### hdb.Client(host, port, username, pwd, op_timeout=15000)
方法描述：
    
    连接远程的HDBServer,并创建hdb.Client对象。  
参数：  
 
     host：HDBServer的IP地址；  
     port: HDBServer的端口；  
     username: 用户名；  
     pwd：密码；  
     op_timeout：连接超时时间，单位毫秒，默认值15000毫秒。  
返回：
    
    hdb.Client对象。
 
#### Client.open_file(path, req_ct_data=False)
方法描述：

    打开服务器端文件。  
参数：  
 
     path：服务端文件的路径，即相对于根目录的文件相对路径，当需要打开内存缓存数据文件时，文件路径需加上前缀memory/  
     req_ct_data: 指定是否需要代码表的data字段(即HCodeInfo的data字段)，默认不需要，如果需要则改参数为True,不需要时候就给默认参数，可以减少网络传输消耗
 
返回：

    hdb.File对象
 
#### Client.close() 
方法描述：
    
    主动关闭client连接，关闭了客户端连接后，再去open_file会抛出runtime error。


### hdb.DB(db_path)
方法描述：
    
    传入本地HDB文件数据库的存放目录，打开该目录，并创建hdb.DB对象。  
参数描述：

    db_path：本地HDB文件数据库的存放目录  

返回：

    hdb.DB对象  

#### DB.create_file(path, ci_type=None, data_types, packet_size=131072)
方法描述：

    在指定目录下创建HDB文件。  
参数描述：

    path: HDB文件的创建路径  
    ci_type：HDB文件中codeinfo数据类型，变量类型为hdb.DataType，默认为None  
    data_types：HDB文件中各种自定义的数据类型，变量类型为List[hdb.DataType]  
    packet_size：HDB文件的块大小。HDB的API读取数据是按块存储，按块读取的。该值越大，读取速度越快，但同时占用的磁盘空间与读取时占用的内存空间也就越大。  
返回：
    
    hdb.File对象  

#### DB.get_trading_days(begin_date, end_date, holiday_file= 'download/holidayinfo.txt')
方法描述：
    
    获取[begin_date,end_date]这个区间内的全部交易日，注意该区间为闭区间。  
参数描述：

    begin_date：开始日期,格式为YYYYMMDD,类型为int,例如：2022年8月10日表示为20220810。
    end_date：结束日期，格式同上。
    holiday_file：记录了交易日历的文件，一般在db_path下面的"download\holidayinfo.txt"路径下。  
返回：

    [begin_date,end_date]之间所有的交易日  

#### DB.open_file(path, mode = 'r')
方法描述：
    
    打开指定的HDB文件  
参数描述：

    path：文件路径。沪深A股与商品期货期权的HDB文件存放于db_path下的marketdata/目录下。  
    mode：支持 'r'和'r+'，'r'表示只读模式，'r+'表示读写模式。  
返回：
    
    hdb.File对象

    

 ### hdb.File 
 对象描述：
     
     无构造方法。该对象只能由调用hdb.DB.openfile、hdb.Client.open_file、hdb.DB.create_file三个方法返回。返回后，会把ci_type、data_types、codetable三个属性的信息读取到本地，供后续使用。

 
 #### File.ci_type
 属性描述：
     
     hdb.DataType对象，用来描述文件中的代码基础信息的数据类型，注意这里描述的是数据类型。  
 
 #### File.data_types
 属性描述：
 
     hdb.DataType对象组成的list，用于描述HDB文件中包含的数据类型。  
 
 #### File.codetable
 属性描述：
 
     hdb.Codetable对象，描述文件中的代码表信息。  
 
 #### File.close()
 方法描述：
     
     关闭文件。  
 
 #### File.get_all_codelists()
 方法描述：
 
     返回文件中的所有代码表的名称，例如：“QSAG”表示沪深A股的全部代码表。  
 
 返回：
     
     代码表名称，str组成的list。  
 
 #### File.get_codelist(cl_name,no_data=True)
 方法描述：
 
     根据代码表名称（cl_name），返回对应的代码表。  
 
 参数：
 
     cl_name：代码表名称cl_name。  
     no_data：是否返回数据，默认不返回数据，即返回的hdb.Codetable对象的data属性为空。  
 
 返回：
     
     hdb.Codetable对象。  
 
 #### File.open_read_task(begin_time = datetime.datetime(1970, 1, 1, 8, 0), end_time = datetime.datetime(1970, 1, 1, 8, 0), symbols, types, offset: int = 0, thread_num = 4)
 方法描述：
 
     建立一个读取任务，读取指定时间段，任意代码列表，任意类型集合的数据。  
 参数：  
 
     begin_time：开始日期时间，datetime类型，默认值为datetime.datetime(1970, 1, 1, 8, 0)。  
     end_time：结束日期时间，datetime类型，默认值为datetim.datetime(1970, 1, 1, 8, 0)。  
     symbols：读取的股票代码列表。例如：```["SH.600000","SZ.000001"]```表示读取“SH.600000”和“SZ.000001”两支股票数据。
     types：读取的数据类型。例如：```["SHStepTrade","SZStepTrade"]```表示读取上海逐笔成交数据与深圳逐笔成交数据。  
     offset：偏移量，默认值为0。
     thread_num：读取线程数，即可以使用多线程读取数据，加快读取速度，默认值为4。  
 
 返回：
 
     hdb.ReadTask对象
 

 #### File. read(symbols, type, fields, begin_time = datetime.datetime(1970, 1, 1, 8, 0), end_time = datetime.datetime(1970, 1, 1, 8, 0), thread_num = 4)
方法描述：
    
    将指定代码列表，指定时间段，某单一类型，指定字段列表的数据读取为hdb.DataFrame。  
参数：

    symbols：代码列表  
    type：数据类型  
    fields：字段列表，该方法可以指定读取特定一些的字段。  
    begin_time：开始日期时间  
    end_time：结束日期时间  
    thread_num：线程数， 即可开启多线程读取数据， 默认值为4。  
返回：

    hdb.DataFrame  

#### File.write(type_id, symbols, local_times, trading_days, data)
方法描述：
    
    将数据写入HDB文件。
参数：

    type_id：类型ID  
    symbols：代码列表  
    local_times：本地时间戳  
    trading_days：交易日  
    data：要写入的数据  

### hdb.ReadTask
对象描述：
    
    无构造方法。该对象只能由调用hdb.File.open_read_task返回。

#### ReadTask.close()
方法描述：

    关闭读取任务。

#### ReadTask.read(max_count=0)
方法描述：
    
    从读取任务中获取读取结果。  
参数：
    
    max_count：最大一次性读取记录条数。  
    
## 附录
    
### code_list_name部分代码列表名称
|选项|说明|
|:-|:-|
|QSAG|全市场A股|
|SHAG|上交所A股|
|SZAG|深交所A股|
|SHAGZT|上海A股涨停|
|SZAGZT|深圳A股涨停|
|QSJJ|全市场基金|
|SHJJ|上海基金|
|SZJJ|深圳基金|
|QSZQ|全市场债券|
|SHZQ|上海债券|
|SZZQ|深圳债券|
|SHNHG|上海逆回购|
|SZNHG|深圳逆回购|
|SHOP|上海期权|
|SZOP|深圳期权|


###  服务器中文件介绍

| 文件夹       | 文件夹中文件名                                               | 介绍                                                         |
| ------------ | ------------------------------------------------------------ | ------------------------------------------------------------ |
| marketdata/  | tick_date(例，tick_20200511)                                 | tick数据，包含CodeInfo、(CodeList)、<br>DataItem(SecurityTick、IndexTick、FuturesTick、OptionsTick、SHStepTrade、SZStepTrade、SZStepOrder、OrderQueueItem、SZOptionsTick) |
| bar/         | day_bar_year(例，day_bar_2020);<br>min_bar_date(例，min_bar_20200511) | K线数据（按日，按分钟）                                      |
| baseinfo/    | SecurityInfo_date(例，SecurityInfo_20200511)                 | 详细的证券代码基本信息数据                                   |
| fundmentals/ | qxdata_year(例，qxdata_2020)                                 | 除息数据                                                     |

### 服务器中Tick数据介绍

| Tick数据       | 名字含义         |
| -------------- | ---------------- |
| CodeList       | 代码列表         |
| CodeInfo       | 代码基本信息     |
| SecurityTick   | 证券Tick数据     |
| IndexTick      | 指数Tick数据     |
| FuturesTick    | 期货Tick数据     |
| OptionsTick    | 期权Tick数据     |
| SHStepTrade    | 上海逐笔成交数据 |
| SZStepTrade    | 深圳逐笔成交数据 |
| SZStepOrder    | 深圳逐笔委托数据 |
| OrderQueueItem | 委托队列数据     |
| SZOptionsTick  | 深圳期权Tick数据 |

## 样列

#### 获取hdb文件中数据类型

In [9]:
import hdb
db = hdb.DB(r"E:\data\bar\min_bar")
file = db.open_file(r"2005\min_bar_20050121", mode="r")
#数据类型数组
data_types = file.data_types
for data_type in data_types:
    print(data_type.name)
###打印各个类型名称

SecurityKdata


In [10]:
#SecurityTick的各个字段
for field in data_types[0].fields:
    print(field.name)

date
time
pre_close
open
high
low
close
volume
turnover
open_interest
pre_settle_price
settle_price


#### 读取代码信息

In [11]:
print("打印类型名：")
print(file.ci_type.name)
print()
print("打印基础信息字段：")
print(file.ci_type.dtype)

打印类型名：
HCodeInfo

打印基础信息字段：
[('sec_type', '<i4'), ('sec_name', 'S24'), ('date', '<i4'), ('high_limited', '<u4'), ('low_limited', '<u4'), ('multiplier', '<i4'), ('margin_ratio', '<i4'), ('price_tick', '<i4'), ('capital', '<i8'), ('cap_change_date', '<u4'), ('trade_date_in', '<u4'), ('trade_date_out', '<u4'), ('is_halt', 'i1'), ('margin_unit', '<u4'), ('margin_ratio_param1', '<i4'), ('margin_ratio_param2', '<i4')]


In [12]:
file.codetable.symbols

['SH.000001',
 'SH.000002',
 'SH.000003',
 'SH.000004',
 'SH.000005',
 'SH.000006',
 'SH.000007',
 'SH.000008',
 'SH.000010',
 'SH.000011',
 'SH.000012',
 'SH.000015',
 'SH.000016',
 'SH.600000',
 'SH.600001',
 'SH.600002',
 'SH.600003',
 'SH.600004',
 'SH.600005',
 'SH.600006',
 'SH.600007',
 'SH.600008',
 'SH.600009',
 'SH.600010',
 'SH.600011',
 'SH.600012',
 'SH.600015',
 'SH.600016',
 'SH.600018',
 'SH.600019',
 'SH.600020',
 'SH.600021',
 'SH.600022',
 'SH.600026',
 'SH.600028',
 'SH.600029',
 'SH.600030',
 'SH.600031',
 'SH.600033',
 'SH.600035',
 'SH.600036',
 'SH.600037',
 'SH.600038',
 'SH.600039',
 'SH.600050',
 'SH.600051',
 'SH.600052',
 'SH.600053',
 'SH.600054',
 'SH.600055',
 'SH.600056',
 'SH.600057',
 'SH.600058',
 'SH.600059',
 'SH.600060',
 'SH.600061',
 'SH.600062',
 'SH.600063',
 'SH.600064',
 'SH.600065',
 'SH.600066',
 'SH.600067',
 'SH.600068',
 'SH.600069',
 'SH.600070',
 'SH.600071',
 'SH.600072',
 'SH.600073',
 'SH.600074',
 'SH.600075',
 'SH.600076',
 'SH.6

In [15]:
index = file.codetable.symbols.index("SH.600000")
file.ci_type.item_data(file.codetable.data[index])

IndexError: index 13 is out of bounds for axis 0 with size 0

In [7]:
import pandas as pd
codetable_pd = pd.DataFrame(file.ci_type.items_data(file.codetable.data))
codetable_pd["sec_name"] = codetable_pd["sec_name"].map(lambda x: x.decode("gbk"))
codetable_pd

,sec_type,sec_name,date,high_limited,low_limited,multiplier,margin_ratio,price_tick,capital,cap_change_date,trade_date_in,trade_date_out,is_halt,margin_unit,margin_ratio_param1,margin_ratio_param2,sec_name_ext
0,3,深证100ETF沽3月2200,20250121,1741,1,0,0,1,0,0,20240725,20250326,0,16050000,120000,70000,b'\xc9\xee\xd6\xa4100ETF\xb9\xc13\xd4\xc22200'
1,3,创业板ETF购3月1650,20250121,8342,94,0,0,1,0,0,20240725,20250326,0,66924000,120000,70000,b'\xb4\xb4\xd2\xb5\xb0\xe5ETF\xb9\xba3\xd4\xc2...
2,3,沪深300ETF沽3月3354A,20250121,2753,1,0,0,1,0,0,20240725,20250326,0,24407900,120000,70000,b'\xbb\xa6\xc9\xee300ETF\xb9\xc13\xd4\xc23354A'
3,3,中证500ETF沽3月2179A,20250121,2814,1,0,0,1,0,0,20240725,20250326,0,72983400,120000,70000,b'\xd6\xd0\xd6\xa4500ETF\xb9\xc13\xd4\xc22179A'
4,3,沪深300ETF沽3月2959A,20250121,1925,1,0,0,1,0,0,20240909,20250326,0,21219800,120000,70000,b'\xbb\xa6\xc9\xee300ETF\xb9\xc13\xd4\xc22959A'
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60409,5,动力定02,0,999999999,10,0,0,10,0,0,20210309,0,0,0,0,0,b'\xb6\xaf\xc1\xa6\xb6\xa802'
60410,5,铜陵定02,20250121,0,0,0,0,0,0,0,20240306,0,0,0,0,0,b'\xcd\xad\xc1\xea\xb6\xa802'
60411,0,,0,0,0,0,0,0,0,0,20250102,20250418,0,0,0,0,b''
60412,0,,0,0,0,0,0,0,0,0,20241101,20250218,0,0,0,0,b''


In [8]:
file.codetable.data

array([2940272635968, 2940272636114, 2940272636260, ..., 2940281455974,
       2940281456120, 2940281456266], shape=(60414,), dtype=uint64)

#### 读取快照数据

In [6]:
#读取SZ.300打头的股票快照数据
task = file.open_read_task(symbols=["SZ.300*"], types=["SecurityTick"])

In [7]:
items = task.read()

In [8]:
#items 为读取出的数据，其中data字段为具体数据的地址
# 需要通过对应的数据类型定义来解析数据
items

array([(b'SZ.300003', 23075, 20220506, 1651798222227, 59, 0, 2098824818800),
       (b'SZ.300004', 23077, 20220506, 1651798222227, 61, 0, 2098824819076),
       (b'SZ.300006', 23078, 20220506, 1651798222227, 62, 0, 2098824819352),
       ...,
       (b'SZ.300978', 24236, 20220506, 1651823945641, 15, 0, 2099802949552),
       (b'SZ.300982', 24239, 20220506, 1651823945641, 18, 0, 2099802949828),
       (b'SZ.300999', 24240, 20220506, 1651823945641, 21, 0, 2099802950104)],
      dtype=[('symbol', 'S24'), ('index', '<i4'), ('trading_day', '<i4'), ('local_time', '<i8'), ('time_point_seq_no', '<i4'), ('type_id', '<i4'), ('data', '<u8')])

In [9]:
type_id = items[0]["type_id"]
file.data_types[type_id].name
#items[0]的type_id字段对应的data_type是SecurityTick

'SecurityTick'

In [10]:
data = file.data_types[type_id].items_data(items["data"])
data

array([( 85021000, 0, 173900,      0,      0,      0, 173900, [     0,      0,      0,      0,      0,      0,      0,      0,      0,      0], [    0,     0,     0,     0,     0,     0,     0,     0,     0,     0], [     0,      0,      0,      0,      0,      0,      0,      0,      0,      0], [     0,      0,      0,      0,      0,      0,      0,      0,      0,      0],     0,        0,         0,       0,       0,      0,      0, 0, 0, 208700, 139100, b'', 182500, 0,          0, b'S0      ', 0),
       ( 85021000, 0,  54700,      0,      0,      0,  54700, [     0,      0,      0,      0,      0,      0,      0,      0,      0,      0], [    0,     0,     0,     0,     0,     0,     0,     0,     0,     0], [     0,      0,      0,      0,      0,      0,      0,      0,      0,      0], [     0,      0,      0,      0,      0,      0,      0,      0,      0,      0],     0,        0,         0,       0,       0,      0,      0, 0, 0,  65600,  43800, b'',      0, 0,          0,

In [11]:
#把data中的字段分别插入dataframe中
#因为dataframe不支持某一个字段是数组类型，所以需要将该字段拆解成多个单独的字段（比如bid_price包含10档数据）
df = pd.DataFrame()
data_type = file.data_types[items[0]["type_id"]]
for name in data_type.dtype.fields:
    field_type = data.dtype.fields[name][0]
    if 0 == field_type.ndim:
        df[name] = data[name]
    elif 1 == field_type.ndim:
        for idx in range(field_type.shape[0]):
            df[name + '.' + str(idx)] = data[name][:, idx]

df['symbol'] = items['symbol']
df['date'] = items['trading_day']
df

,time,status,pre_close,open,high,low,match,ask_price.0,ask_price.1,ask_price.2,...,high_limited,low_limited,prefix,syl1,syl2,sd2,trading_phase_code,pre_iopv,symbol,date
0,85021000,0,173900,0,0,0,173900,0,0,0,...,208700,139100,b'',182500,0,0,b'S0 ',0,b'SZ.300003',20220506
1,85021000,0,54700,0,0,0,54700,0,0,0,...,65600,43800,b'',0,0,0,b'S0 ',0,b'SZ.300004',20220506
2,85021000,0,40600,0,0,0,40600,0,0,0,...,48700,32500,b'',0,0,0,b'S0 ',0,b'SZ.300006',20220506
3,85021000,0,148800,0,0,0,148800,0,0,0,...,178600,119000,b'',183400,0,0,b'S0 ',0,b'SZ.300007',20220506
4,85021000,0,60100,0,0,0,60100,0,0,0,...,72100,48100,b'',0,0,0,b'S0 ',0,b'SZ.300013',20220506
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3543950,155903000,0,169700,163000,170100,162600,167500,167600,167700,167800,...,203600,135800,b'',389300,0,-171798792,b'E0 ',0,b'SZ.300967',20220506
3543951,155903000,0,333100,326700,335000,322500,327400,327800,328000,328200,...,399700,266500,b'',187900,0,-171798892,b'E0 ',0,b'SZ.300977',20220506
3543952,155903000,0,108600,105100,110000,105100,110000,110000,110100,110300,...,130300,86900,b'',401300,0,1400,b'E0 ',0,b'SZ.300978',20220506
3543953,155903000,0,395200,382200,392500,378800,385400,385400,386000,386200,...,474200,316200,b'',179600,0,400,b'E0 ',0,b'SZ.300982',20220506


In [12]:
#将上述过程封装后可以调用read_data
def read_data(f, symbols, data_type, begin=None, end=None):
    if (begin is None) and (end is None):
        task = f.open_read_task(symbols=symbols, types=[data_type])
    elif begin is None:
        task = f.open_read_task(end_time=end, symbols=symbols, types=[data_type])
    elif end is None:
        task = f.open_read_task(begin_time=begin, symbols=symbols, types=[data_type])
    else:
        task = f.open_read_task(begin_time=begin, end_time=end, symbols=symbols, types=[data_type])
    items = task.read()
    data_type = f.data_types[items['type_id'][0]]
    data = data_type.items_data(items['data'])
    df = pd.DataFrame()
    for name in data_type.dtype.fields:
        field_type = data.dtype.fields[name][0]
        if 0 == field_type.ndim:
            df[name] = data[name]
        elif 1 == field_type.ndim:
            for idx in range(field_type.shape[0]):
                df[name + '.' + str(idx)] = data[name][:, idx]
    task.close()
    df['symbol'] = items['symbol']
    df['date'] = items['trading_day']
    return df

data = read_data(file, ["SZ.300*"], "SecurityTick")

In [13]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3543955 entries, 0 to 3543954
Data columns (total 66 columns):
 #   Column                  Dtype 
---  ------                  ----- 
 0   time                    int32 
 1   status                  int32 
 2   pre_close               uint32
 3   open                    uint32
 4   high                    uint32
 5   low                     uint32
 6   match                   uint32
 7   ask_price.0             uint32
 8   ask_price.1             uint32
 9   ask_price.2             uint32
 10  ask_price.3             uint32
 11  ask_price.4             uint32
 12  ask_price.5             uint32
 13  ask_price.6             uint32
 14  ask_price.7             uint32
 15  ask_price.8             uint32
 16  ask_price.9             uint32
 17  ask_vol.0               uint32
 18  ask_vol.1               uint32
 19  ask_vol.2               uint32
 20  ask_vol.3               uint32
 21  ask_vol.4               uint32
 22  ask_vol.5         

In [14]:
data.head()

,time,status,pre_close,open,high,low,match,ask_price.0,ask_price.1,ask_price.2,...,high_limited,low_limited,prefix,syl1,syl2,sd2,trading_phase_code,pre_iopv,symbol,date
0,85021000,0,173900,0,0,0,173900,0,0,0,...,208700,139100,b'',182500,0,0,b'S0 ',0,b'SZ.300003',20220506
1,85021000,0,54700,0,0,0,54700,0,0,0,...,65600,43800,b'',0,0,0,b'S0 ',0,b'SZ.300004',20220506
2,85021000,0,40600,0,0,0,40600,0,0,0,...,48700,32500,b'',0,0,0,b'S0 ',0,b'SZ.300006',20220506
3,85021000,0,148800,0,0,0,148800,0,0,0,...,178600,119000,b'',183400,0,0,b'S0 ',0,b'SZ.300007',20220506
4,85021000,0,60100,0,0,0,60100,0,0,0,...,72100,48100,b'',0,0,0,b'S0 ',0,b'SZ.300013',20220506


#### 读取逐笔成交数据

In [15]:
['SecurityTick',
 'IndexTick',
 'FuturesTick',
 'OptionsTick',
 'SHStepTrade',
 'SZStepTrade',
 'SZStepOrder',
 'OrderQueueItem',
 'SZOptionsTick',
 'SHStepOrder',
 'FPSHStepTrade']

['SecurityTick',
 'IndexTick',
 'FuturesTick',
 'OptionsTick',
 'SHStepTrade',
 'SZStepTrade',
 'SZStepOrder',
 'OrderQueueItem',
 'SZOptionsTick',
 'SHStepOrder',
 'FPSHStepTrade']

In [16]:
#将上述过程封装后可以调用read_data
data = read_data(file, ["SZ.300*"], "SZStepTrade")
data.head()

,channel_no,appl_seq_num,md_stream_id,bid_appl_seq_num,offer_appl_seq_num,security_id,security_id_source,last_px,last_qty,exec_type,transact_time,symbol,date
0,2011,538,b'011',0,43,b'300481 ',b'102 ',0,2000,52,91500030,b'SZ.300481',20220506
1,2011,1153,b'011',1152,0,b'300446 ',b'102 ',0,100,52,91500030,b'SZ.300446',20220506
2,2014,1795,b'011',1794,0,b'300857 ',b'102 ',0,1300,52,91500030,b'SZ.300857',20220506
3,2012,3682,b'011',0,3677,b'300438 ',b'102 ',0,5200,52,91500030,b'SZ.300438',20220506
4,2012,3697,b'011',0,3696,b'300438 ',b'102 ',0,5200,52,91500030,b'SZ.300438',20220506


In [17]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18719104 entries, 0 to 18719103
Data columns (total 13 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   channel_no          uint16
 1   appl_seq_num        int64 
 2   md_stream_id        |S3   
 3   bid_appl_seq_num    int64 
 4   offer_appl_seq_num  int64 
 5   security_id         |S8   
 6   security_id_source  |S4   
 7   last_px             int64 
 8   last_qty            int64 
 9   exec_type           int8  
 10  transact_time       int64 
 11  symbol              |S24  
 12  date                int32 
dtypes: bytes192(1), bytes24(1), bytes32(1), bytes64(1), int32(1), int64(6), int8(1), uint16(1)
memory usage: 1.6 GB


In [18]:
file.close()

### 获取k线数据及复权数据
#### 获取不复权k线数据

In [19]:
# 该函数可以获取不复权的k线
def load_min_bar_from_hdb(symbols,
                          db_path,
                          start,
                          end):
    start_date = int(start.strftime("%Y%m%d"))
    end_date = int(end.strftime("%Y%m%d"))
    db = hdb.DB(db_path)
    total_dates = db.get_trading_days(start_date, end_date)
    load_year = None
    load_years = []
    for cur_date in total_dates:
        cur_year = cur_date // 10000
        if cur_year != load_year:
            load_year = cur_year
            load_years.append(str(cur_year))
    ret = []
    for cur_year in load_years:
        file_name = "%s_%s" % ("bar/day_bar", cur_year)
        file = db.open_file(file_name)
        data = read_data(file, symbols, "SecurityKdata", start, end)
        ret.append(data)
        file.close()
    kline_data = pd.concat(ret)
    kline_data["symbol"] = kline_data["symbol"].str.decode("utf8")
#    kline_data["date"] = pd.to_datetime(kline_data["date"], format="%Y%m%d")
    return kline_data

In [21]:
import datetime
start = datetime.date(2022,11,1)
end = datetime.date(2023,1,31)
symbols = ["SZ.123119","SH.600085"]
db_path = "Z:/hdb_data"
kline_data = load_min_bar_from_hdb(symbols, db_path, start, end)
kline_data

,date,time,pre_close,open,high,low,close,volume,turnover,open_interest,pre_settle_price,settle_price,symbol
0,20221101,150000,1255000,1248100,1291790,1248100,1283460,654877,83368647,0,0,0,SZ.123119
1,20221101,150000,487500,476000,481800,462800,473000,22655583,1065165972,0,0,0,SH.600085
2,20221102,150000,1283460,1285400,1321780,1283450,1318980,2387949,312538036,0,0,0,SZ.123119
3,20221102,150000,473000,482500,515100,474500,495800,25116193,1241642059,0,0,0,SH.600085
4,20221103,150000,1318980,1307340,1325260,1300000,1309380,994107,130432348,0,0,0,SZ.123119
...,...,...,...,...,...,...,...,...,...,...,...,...,...
25,20230119,150000,459300,459400,470800,458000,466500,7573774,352756964,0,0,0,SH.600085
26,20230120,150000,1316000,1314180,1330000,1307250,1318590,320039,42281249,0,0,0,SZ.123119
27,20230120,150000,466500,466300,478100,466000,469800,9153584,432152230,0,0,0,SH.600085
28,20230130,150000,1318590,1319400,1345000,1319400,1329990,577822,76966278,0,0,0,SZ.123119


#### 获取复权因子
load_adj_factor(data_base_path, symbols, begin_date, end_date, fq, is_from_initial=False) 
  
方法描述：
    
    该方法可以获取指定股票列表，日期在闭区间[begin_date, end_date]内，每天每支标的的复权因子。复权因子乘以不复权价即可得对应的复权价。
    
参数描述：

    data_base_path：HDB目录
    symbols：股票代码列表
    begin_date：开始日期，格式：yyyymmdd，例如20220101
    end_date：结束日期，格式同上
    fq：复权方式，前复权填“pre”，后复权填“post”。
    is_from_initial：后复权时，复权价是否从上市首日开始计算，默认值为False
    
返回：每只标的，在闭区间[begin_date, end_date]内的所有复权因子

In [22]:
def load_adj_factor(data_base_path, symbols, begin_date, end_date, fq, is_from_initial=False):
    db = hdb.DB(data_base_path)
    total_dates = db.get_trading_days(begin_date, end_date)
    load_year = None
    load_years = []
    is_from_initial
    for cur_date in total_dates:
        cur_year = cur_date // 10000
        if cur_year != load_year:
            load_year = cur_year
            load_years.append(cur_year)
    adj_factors = dict()
    adj_data = dict()
    for cur_year in load_years:
        file_path = "bar\day_bar_%s" % cur_year
        file = db.open_file(file_path, mode="r")
        for symbol in symbols:
            if is_from_initial:
                index =  file.codetable.symbols.index(symbol)
                codeinfo = file.ci_type.item_data(file.codetable.data[index])
                bdate = codeinfo['date'][0]        
                begin_time = max(datetime.datetime(cur_year,1,1,0),datetime.datetime.strptime(str(bdate),"%Y%m%d"))
                if not(symbol in adj_factors.keys()):
                    adj_factors[symbol] = (codeinfo["capital"] / codeinfo["multiplier"])[0]   
            else:
                begin_time = max(datetime.datetime(cur_year,1,1,0),datetime.datetime.strptime(str(begin_date),"%Y%m%d"))
                if not(symbol in adj_factors.keys()):
                    adj_factors[symbol] = 1
            end_time = min(datetime.datetime(cur_year,12,31,23,59,59),datetime.datetime.strptime(str(end_date),"%Y%m%d"))
            data = file.read([symbol],"SecurityKdata",["date","pre_close","close"],begin_time,end_time)
            data_pd = hdb.util.convert_hdbframe_to_pandasframe(data, ["date","pre_close","close"])
            if symbol in adj_data.keys():
                adj_data[symbol] = pd.concat([adj_data[symbol], data_pd])
            else:
                adj_data[symbol] = data_pd
    qfactor = dict()
    for symbol in symbols:
        daybar = adj_data[symbol]
        if fq == "pre":
            daybar["fq_factor"] = daybar["pre_close"]/daybar["close"]
            close = daybar["close"].iloc[-1]
            daybar["fq_factor"] = daybar["fq_factor"].iloc[::-1].cumprod()[::-1]
            daybar["fq_factor"] = daybar["fq_factor"]*close
            daybar["fq_factor"] = daybar["fq_factor"]/daybar["pre_close"]
        elif fq == "post":
            adj_factor = adj_factors[symbol]
            daybar["fq_factor"] = daybar["close"]/daybar["pre_close"]
            daybar["fq_factor"] = daybar["fq_factor"].cumprod()
            pre_close = daybar["pre_close"].iloc[0]
            daybar["fq_factor"] = daybar["fq_factor"]*pre_close*adj_factor
            daybar["fq_factor"] = daybar["fq_factor"]/daybar["close"]
            daybar = daybar[daybar["date"]>=begin_date]
#        qfactor[symbol] = daybar
        qfactor[symbol] = daybar.groupby("date")["fq_factor"].first().to_dict()
    return qfactor

In [23]:
db_path = "Z:/hdb_data"
symbols = ["SH.600085","SZ.000001"]
begin_date = 20210801
end_date = 20230310
fq = "post"
factor = load_adj_factor(db_path, symbols, begin_date, end_date, fq, is_from_initial=True) 
factor

{'SH.600085': {20210802: 11.430051000000002,
  20210803: 11.430051,
  20210804: 11.430051,
  20210805: 11.430051,
  20210806: 11.430051,
  20210809: 11.430051,
  20210810: 11.430051,
  20210811: 11.430050999999999,
  20210812: 11.520130211033274,
  20210813: 11.520130211033274,
  20210816: 11.520130211033273,
  20210817: 11.52013021103327,
  20210818: 11.52013021103327,
  20210819: 11.520130211033269,
  20210820: 11.520130211033269,
  20210823: 11.520130211033269,
  20210824: 11.52013021103327,
  20210825: 11.520130211033273,
  20210826: 11.520130211033273,
  20210827: 11.520130211033273,
  20210830: 11.520130211033269,
  20210831: 11.52013021103327,
  20210901: 11.520130211033269,
  20210902: 11.520130211033269,
  20210903: 11.520130211033269,
  20210906: 11.520130211033269,
  20210907: 11.520130211033269,
  20210908: 11.520130211033267,
  20210909: 11.520130211033269,
  20210910: 11.520130211033269,
  20210913: 11.520130211033269,
  20210914: 11.520130211033266,
  20210915: 11.520130

In [24]:
#计算sh600085的k线的收盘价的后复权价
sh600085 = kline_data[kline_data["symbol"]=="SH.600085"].copy()
sh600085['close_fq'] = sh600085.apply(lambda x: int(round(factor['SH.600085'][x['date']]*x['close'],-2)),axis=1)
sh600085.head()

,date,time,pre_close,open,high,low,close,volume,turnover,open_interest,pre_settle_price,settle_price,symbol,close_fq
1,20221101,150000,487500,476000,481800,462800,473000,22655583,1065165972,0,0,0,SH.600085,5481800
3,20221102,150000,473000,482500,515100,474500,495800,25116193,1241642059,0,0,0,SH.600085,5746000
5,20221103,150000,495800,490300,495400,480800,486800,11975890,583286659,0,0,0,SH.600085,5641700
7,20221104,150000,486800,483200,506700,481500,498000,11213095,554260198,0,0,0,SH.600085,5771500
9,20221107,150000,498000,499800,500000,486000,489800,11446232,561484304,0,0,0,SH.600085,5676500
